# Phần 1 — Câu hỏi Trắc nghiệm
Nhóm: E_DATA

##  Thiết lập thư viện

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime


## Load dữ liệu

In [2]:
# Đọc tất cả file dataset
products         = pd.read_csv('products.csv')
customers        = pd.read_csv('customers.csv')
promotions       = pd.read_csv('promotions.csv')
geography        = pd.read_csv('geography.csv')
payments         = pd.read_csv('payments.csv')
shipments        = pd.read_csv('shipments.csv')
returns          = pd.read_csv('returns.csv')
reviews          = pd.read_csv('reviews.csv')
sales_train      = pd.read_csv('sales.csv')          # train
sample_submission = pd.read_csv('sample_submission.csv')
inventory        = pd.read_csv('inventory.csv')
web_traffic      = pd.read_csv('web_traffic.csv')
orders           = pd.read_csv('orders.csv', parse_dates=['order_date'])

# promo_id dạng string để tránh mất leading zeros
order_items = pd.read_csv('order_items.csv', dtype={'promo_id': str, 'promo_id_2': str})


In [3]:
# Kiểm tra nhanh shape, duplicates, nulls
for name, df in [('orders', orders), ('order_items', order_items),
                  ('products', products), ('returns', returns)]:
    print(f'\n=== {name} ===')
    print(f'  Shape     : {df.shape}')
    print(f'  Duplicates: {df.duplicated().sum()}')
    nulls = df.isna().sum()
    nulls = nulls[nulls > 0]
    print(f'  Nulls     :\n{nulls if not nulls.empty else "  (none)"}' )



=== orders ===
  Shape     : (646945, 8)
  Duplicates: 0
  Nulls     :
  (none)

=== order_items ===
  Shape     : (714669, 7)
  Duplicates: 0
  Nulls     :
promo_id      438353
promo_id_2    714463
dtype: int64

=== products ===
  Shape     : (2412, 8)
  Duplicates: 0
  Nulls     :
  (none)

=== returns ===
  Shape     : (39939, 7)
  Duplicates: 0
  Nulls     :
  (none)


---
## Bài làm

### Q1.

Trong số các khách hàng có nhiều hơn một đơn hàng, **trung vị số ngày giữa hai lần mua liên tiếp** (inter-order gap) xấp xỉ là bao nhiêu? (Tính từ `orders.csv`)

A) 30 ngày

B) 90 ngày

C) 144 ngày 

D) 365 ngày

In [7]:
# Lọc khách có > 1 đơn
multi_buyers = orders.groupby('customer_id').filter(lambda x: len(x) > 1)

# Sắp xếp theo customer_id và order_date
multi_buyers = multi_buyers.sort_values(['customer_id', 'order_date'])

# Tính khoảng cách ngày giữa các đơn liên tiếp
gaps = multi_buyers.groupby('customer_id')['order_date'].diff().dropna().dt.days

#Trung vị
print(f'Median inter-order gap: {gaps.median():.0f} ngày') 

Median inter-order gap: 144 ngày


**Đáp án: C**

### Q2. 

Phân khúc sản phẩm (segment) nào trong `products.csv` có **tỷ suất lợi nhuận gộp
trung bình** cao nhất, với công thức (price − cogs)/price?

 A) Premium
 
 B) Performance
 
 C) Activewear
 
 D) Standard 

In [8]:
products['gross_margin'] = (products['price'] - products['cogs']) / products['price']
margin_by_segment = (
    products.groupby('segment')['gross_margin']
    .mean()
    .sort_values(ascending=False)
    .round(4)
)
print(margin_by_segment)

segment
Standard       0.3134
Premium        0.2854
All-weather    0.2842
Activewear     0.2656
Performance    0.2636
Balanced       0.2580
Trendy         0.2408
Everyday       0.2363
Name: gross_margin, dtype: float64


**Đáp án: D**

### Q3. 

Trong các bản ghi trả hàng liên kết với sản phẩm thuộc danh mục **Streetwear** (join `returns` với `products` theo `product_id`), lý do trả hàng nào xuất hiện **nhiều nhất**?

A) defective
B) wrong_size
C) changed_mind
D) not_as_described

In [9]:
# Join returns với products theo product_id
returns_products = returns.merge(products[['product_id', 'category']], on='product_id')
streetwear_returns = returns_products[returns_products['category'] == 'Streetwear']
reason_counts = streetwear_returns['return_reason'].value_counts()
print(reason_counts)

return_reason
wrong_size          7626
defective           4330
not_as_described    3854
changed_mind        3830
late_delivery       2159
Name: count, dtype: int64


** Đáp án: B**

### Q4.

Trong `web_traffic.csv`, nguồn truy cập (traffic_source) nào có **tỷ lệ thoát trung bình (bounce_rate) thấp nhất**?

 A) organic_search
 
 B) paid_search
  
 C) email_campaign

 D) social_media

In [10]:
avg_bounce = (
    web_traffic.groupby('traffic_source')['bounce_rate']
    .mean()
    .sort_values())
print(avg_bounce)

traffic_source
email_campaign    0.004458
social_media      0.004476
paid_search       0.004478
referral          0.004499
organic_search    0.004504
direct            0.004511
Name: bounce_rate, dtype: float64


** Đáp án: C**

### Q5.

Tỷ lệ phần trăm các dòng trong `order_items.csv` có áp dụng khuyến mãi (tức là promo_id không null) xấp xỉ là bao nhiêu?

 A) 12%
 
 B) 25%
 
 C) 39% 
 
 D) 54%

In [15]:
pct = order_items['promo_id'].notna().mean() * 100
print(f'{pct:.0f}%')


39%


** Đáp án: C**

### Q6. 

Trong `customers.csv`, xét các khách hàng có `age_group` khác null, nhóm tuổi nào có **số đơn hàng trung bình trên mỗi khách hàng cao nhất**? *(tổng số đơn / số khách hàng trong nhóm)*

 A) 55+
  
 B) 25–34

 C) 35–44

 D) 45–54

In [16]:
#đếm số đơn của mỗi khách: mỗi khách hàng, đếm có bao nhiêu dòng trong orders -> ra số đơn
orders_per_cust = orders.groupby('customer_id').size().reset_index(name='n_orders')
#lọc khách hàng có age_group,join với số đơn
cust_age = customers[customers['age_group'].notna()].merge(
    orders_per_cust, on='customer_id', how='left'
)
cust_age['n_orders'] = cust_age['n_orders'].fillna(0) #khách hàng chưa mua lần nào NA -> thay = 0
#tính trung bình
avg_orders = cust_age.groupby('age_group')['n_orders'].mean().sort_values(ascending=False)
print(avg_orders.round(2))

age_group
55+      5.41
45-54    5.36
35-44    5.34
25-34    5.25
18-24    5.23
Name: n_orders, dtype: float64


**Đáp án: A**

### Q7. 

Vùng (region) nào trong `geography.csv` tạo ra **tổng doanh thu cao nhất** trong `sales_train.csv`?

 A) West
  
 B) Central
  
 C) East
  
 D) Cả ba vùng xấp xỉ bằng nhau

In [19]:
# Tổng doanh thu theo vùng: join orders -> zip -> geography
orders_geo = orders.merge(geography, on='zip')
# Lấy revenue từ payments hoặc tính từ order_items
# Giả sử dùng payments.payment_value
order_revenue = payments[['order_id', 'payment_value']]
orders_geo_rev = orders_geo.merge(order_revenue, on='order_id')
region_revenue = orders_geo_rev.groupby('region')['payment_value'].sum().sort_values(ascending=False)
print(region_revenue)

region
East       7.291151e+09
Central    4.719491e+09
West       3.670227e+09
Name: payment_value, dtype: float64


**Đáp án: C**

### Q8. Phương thức thanh toán phổ biến nhất trong đơn huỷ
Trong các đơn hàng có order_status = 'cancelled', **phương thức thanh toán nào** được sử dụng nhiều nhất?

 A) credit_card 
 
 B) cod
 
 C) paypal
 
 D) bank_transfer

In [20]:
cancelled = orders[orders['order_status'] == 'cancelled']
cancelled['payment_method'].value_counts()

payment_method
credit_card      28452
cod              15468
paypal            7817
apple_pay         5190
bank_transfer     2535
Name: count, dtype: int64

**Đáp án: A**

### Q9. 

Trong bốn kích thước sản phẩm (S, M, L, XL), kích thước nào có **tỷ lệ trả hàng cao nhất**, được định nghĩa là số bản ghi trong returns chia cho số dòng trong order_items (join với products theo product_id)?**


A) S

B) M

C) L

D) XL

In [22]:
# Lấy size từ products: 
items_size = order_items.merge(products[['product_id', 'size']], on='product_id')

# Đếm số dòng order_items theo size, size(): đếm tất cả dòng kể cả null
item_counts = items_size.groupby('size').size().reset_index(name='total_items')

# Đếm số returns theo product_id, rồi theo size
return_counts = returns.merge(
    products[['product_id', 'size']], on='product_id').groupby('size').size().reset_index(name='total_returns')
# Merge và tính tỷ lệ
ratio = return_counts.merge(item_counts, on='size')
ratio['return_rate'] = ratio['total_returns'] / ratio['total_items']
print(ratio[['size', 'return_rate']].sort_values('return_rate', ascending=False))

  size  return_rate
2    S     0.056515
0    L     0.056250
1    M     0.055660
3   XL     0.055200


**Đáp án: A**

### Q10. 

Trong payments.csv, kế hoạch trả góp nào có giá trị thanh toán trung bình trên
mỗi đơn hàng cao nhất?**

A) 1 kỳ (trả một lần)

B) 3 kỳ

C) 6 kỳ

D) 12 kỳ

In [24]:
avg_payment = (
    payments
    .groupby('installments')['payment_value']
    .mean()
    .round(2)
    .sort_values(ascending=False))
avg_payment

installments
6     24446.65
3     24399.64
12    24245.77
1     24113.27
2       708.47
Name: payment_value, dtype: float64

**Đáp án: C**